In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, TimestampType
from functools import reduce

LANDING = "/Volumes/ifood_case/bronze/landing"

def load_bronze(taxi_type: str, pickup_col: str, dropoff_col: str):
    files = [f.path for f in dbutils.fs.ls(f"{LANDING}/{taxi_type}")]
    dfs = []
    for path in files:
        df = (
            spark.read.parquet(path)
            .withColumn("VendorID", F.col("VendorID").cast(IntegerType()))
            .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType()))
            .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
            .withColumn(pickup_col, F.col(pickup_col).cast(TimestampType()))
            .withColumn(dropoff_col, F.col(dropoff_col).cast(TimestampType()))
            .select("VendorID", "passenger_count", "total_amount", pickup_col, dropoff_col)
            .withColumn("_source_file", F.lit(path.split("/")[-1]))
            .withColumn("_ingestion_ts", F.current_timestamp())
        )
        dfs.append(df)
    return reduce(lambda a, b: a.unionByName(b), dfs)

bronze_yellow = load_bronze("yellow", "tpep_pickup_datetime", "tpep_dropoff_datetime")
bronze_green  = load_bronze("green",  "lpep_pickup_datetime",  "lpep_dropoff_datetime")

(bronze_yellow.write.format("delta").mode("overwrite")
    .saveAsTable("ifood_case.bronze.yellow_taxi_trips"))
(bronze_green.write.format("delta").mode("overwrite")
    .saveAsTable("ifood_case.bronze.green_taxi_trips"))

In [0]:
%sql
SELECT _source_file, COUNT(*) FROM ifood_case.bronze.yellow_taxi_trips GROUP BY 1;